# High-cardinality Encoding Comparison

Reproduce the existing listing-only LightGBM using ordinal encoding for both categorical groups. Then compare ordinal-as-numeric, training frequency, and native categorical encodings on the same E2.5 core features, with low-cardinality one-hot encoding. No scaling, tuning, early stopping, or test data are used.

Native categories use training-only unordered pandas categories and explicit LightGBM categorical inputs; unknown validation categories become missing. Frequency uses training-row proportions and zero for unseen values. Ordinal values remain numeric; unknown values are -1. Numerical NaNs are retained for LightGBM.

The minimum validation RMSE determines one fixed method for all subsequent E2 and future E3 experiments. This is validation-based selection, not an independent final generalization estimate.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'src').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from experiments.feature_engineering.utils.core import *
from IPython.display import display, Image


In [2]:
comparison, reference_audit = stage1()
display(pd.read_csv(OUT / 'listing_reference.csv'))
display(reference_audit)
display(comparison)
print((AUDITS / 'selected_encoding.json').read_text())

Listing reference ordinal {'RMSE': 480.600349, 'MAE': 357.09777, 'R2': 0.467661, 'Mean Error': -42.015405}


/Users/wangyufan/Desktop/master/CS5228/project/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


ordinal ordinal {'RMSE': 479.496863, 'MAE': 356.508969, 'R2': 0.470103, 'Mean Error': -45.033072}


/Users/wangyufan/Desktop/master/CS5228/project/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


frequency frequency {'RMSE': 479.805553, 'MAE': 357.295635, 'R2': 0.469421, 'Mean Error': -47.312233}


...

native native {'RMSE': 482.634963, 'MAE': 359.682289, 'R2': 0.463145, 'Mean Error': -47.311319}


,experiment,encoding,low_card_encoding,raw_features,raw_feature_count,encoded_features,feature_count,categorical_features,model_params,train_rows,validation_rows,encoding_seconds,training_seconds,RMSE,MAE,R2,Mean Error
0,Listing reference,ordinal,ordinal,"[""FLOOR_AREA_SQM"", ""flat_age"", ""month_index"", ...",8,"[""FLOOR_AREA_SQM"", ""flat_age"", ""month_index"", ...",8,[],"{""learning_rate"": 0.05, ""n_estimators"": 500, ""...",117777,31582,0.124144,2.21563,480.600349,357.09777,0.467661,-42.015405


{'notebook_rmse': 480.6,
 'csv_rmse': 483.0913728290935,
 'reproduced_rmse': 480.60034889314807,
 'notebook_match_2dp': True,
 'csv_match': False,
 'python': '3.13.7',
 'versions': {'numpy': '2.2.4',
  'pandas': '2.2.3',
  'scikit-learn': '1.6.1',
  'lightgbm': '4.6.0'},
 'train_file_sha256': '6d7a417de72112361f8353659ba6fe44a442b07465b488aff1b03f7ec1ea3ba6'}

,experiment,encoding,low_card_encoding,raw_features,raw_feature_count,encoded_features,feature_count,categorical_features,model_params,train_rows,validation_rows,encoding_seconds,training_seconds,RMSE,MAE,R2,Mean Error,control,delta_rmse,relative_improvement_pct
0,ordinal,ordinal,onehot,"[""FLOOR_AREA_SQM"", ""flat_age"", ""month_index"", ...",8,"[""FLOOR_AREA_SQM"", ""flat_age"", ""month_index"", ...",53,[],"{""learning_rate"": 0.05, ""n_estimators"": 500, ""...",117777,31582,0.197503,2.215994,479.496863,356.508969,0.470103,-45.033072,ordinal,0.00000,0.000000
1,frequency,frequency,onehot,"[""FLOOR_AREA_SQM"", ""flat_age"", ""month_index"", ...",8,"[""FLOOR_AREA_SQM"", ""flat_age"", ""month_index"", ...",53,[],"{""learning_rate"": 0.05, ""n_estimators"": 500, ""...",117777,31582,0.160186,2.139831,479.805553,357.295635,0.469421,-47.312233,ordinal,-0.30869,-0.064378
2,native,native,onehot,"[""FLOOR_AREA_SQM"", ""flat_age"", ""month_index"", ...",8,"[""FLOOR_AREA_SQM"", ""flat_age"", ""month_index"", ...",53,"[""BLOCK"", ""STREET""]","{""learning_rate"": 0.05, ""n_estimators"": 500, ""...",117777,31582,0.161081,2.561095,482.634963,359.682289,0.463145,-47.311319,ordinal,-3.13810,-0.654457


{
  "method": "ordinal",
  "selection": "Lowest validation RMSE on E2.5; fixed for all E2 and future E3."
}


The stored model-comparison notebook reports LightGBM RMSE 480.60; the adjacent legacy CSV reports 483.09. The audit checks both separately and never overwrites either. Runtime measures fit only, with encoding time reported separately. Raw and encoded feature lists, model parameters, counts, and four finite metrics are exported to CSV.